In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q15_rewrite_small/checkpoints/pre_cell_2.pickle

In [4]:
%%RecordEvent
%%time
### cell 2 ###

# cell 2
# Define date bounds as strings to push comparisons to the GPU
date_start = "1996-01-01"
date_end = "1996-04-01"

# Filter by ship date entirely on the GPU
tmp = lineitem[(lineitem.L_SHIPDATE >= date_start) & (lineitem.L_SHIPDATE < date_end)]

# Compute revenue parts
# Select only needed columns before the multiply to reduce memory footprint
tmp = tmp[["L_SUPPKEY", "L_EXTENDEDPRICE", "L_DISCOUNT"]]
tmp["REVENUE_PARTS"] = tmp.L_EXTENDEDPRICE * (1 - tmp.L_DISCOUNT)

# Aggregate GPU-side and reset index with new column name
revenue_table = (
    tmp.groupby("L_SUPPKEY").REVENUE_PARTS.sum()
    .reset_index(name="TOTAL_REVENUE")
    .rename(columns={"L_SUPPKEY": "SUPPLIER_NO"})
)

# Find the maximum revenue and filter
max_rev = revenue_table.TOTAL_REVENUE.max()
revenue_table = revenue_table[revenue_table.TOTAL_REVENUE == max_rev]

# Join with supplier details and select final columns
total = (
    supplier[["S_SUPPKEY", "S_NAME", "S_ADDRESS", "S_PHONE"]]
    .merge(revenue_table, left_on="S_SUPPKEY", right_on="SUPPLIER_NO")
    [["S_SUPPKEY", "S_NAME", "S_ADDRESS", "S_PHONE", "TOTAL_REVENUE"]]
)


CPU times: user 74.2 ms, sys: 40.8 ms, total: 115 ms
Wall time: 135 ms


In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/rewritten/o4_mini_high_small/checkpoints/post_cell_2_try_0.pickle